# Lung Cancer Prediction using Machine Learning & Deep Learning

**Project by: N. Mohammed Sohaib (74)**  
**PBEL NASCOM Internship – AI/ML Project**

---

## Objective
Predict whether a patient is at risk of lung cancer based on their symptoms, habits, and health indicators using multiple ML models and a Deep Learning (ANN) model.

## Dataset
Lung Cancer Patient Survey Dataset — 1,000 patient records with 15 features (age, smoking habits, symptoms, etc.) and a binary target (Lung Cancer: YES/NO).

## Steps
1. Import Libraries
2. Load & Explore Data (EDA)
3. Data Preprocessing
4. Data Visualization
5. Build ML Models (Logistic Regression, Decision Tree, Random Forest, SVM, KNN)
6. Build ANN (Deep Learning) Model
7. Compare All Models
8. Save Best Model

## Step 1: Install & Import Libraries

In [6]:
!pip install scikit-learn tensorflow matplotlib seaborn

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 2: Load & Explore Data (EDA)

In [ ]:
df = pd.read_csv("lung_cancer_data.csv")
print(f"Dataset Shape: {df.shape}")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1] - 1}")
print(f"\nColumn Names:")
print(df.columns.tolist())
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'Project/Lung_Cancer_Prediction/lung_cancer_data.csv'

In [ ]:
# Dataset Info
print("Dataset Info:")
print("=" * 50)
df.info()
print(f"\nMissing Values:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Statistical Summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Target Distribution
print("Target Variable Distribution:")
print(df['LUNG_CANCER'].value_counts())
print(f"\nPercentage:")
print(df['LUNG_CANCER'].value_counts(normalize=True).round(3) * 100)

## Step 3: Data Preprocessing

In [ ]:
# Encode categorical variables
le_gender = LabelEncoder()
df['GENDER'] = le_gender.fit_transform(df['GENDER'])  # M=1, F=0

le_cancer = LabelEncoder()
df['LUNG_CANCER'] = le_cancer.fit_transform(df['LUNG_CANCER'])  # YES=1, NO=0

print("Gender encoding:", dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))
print("Cancer encoding:", dict(zip(le_cancer.classes_, le_cancer.transform(le_cancer.classes_))))
print(f"\nDataset after encoding:")
df.head()

In [ ]:
# Separate Features and Target
X = df.drop('LUNG_CANCER', axis=1)
y = df['LUNG_CANCER']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")

In [ ]:
# Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature scaling applied (StandardScaler)")
print(f"Mean after scaling: {X_scaled.mean(axis=0).round(2)}")
print(f"Std after scaling: {X_scaled.std(axis=0).round(2)}")

In [ ]:
# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"\nTraining target distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nTesting target distribution:")
print(pd.Series(y_test).value_counts())

## Step 4: Data Visualization

In [ ]:
# Target Distribution Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
colors = ['#2ecc71', '#e74c3c']
df['LUNG_CANCER'].value_counts().plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Lung Cancer Distribution', fontsize=14)
axes[0].set_xlabel('Lung Cancer (0=NO, 1=YES)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['NO', 'YES'], rotation=0)

# Pie chart
df['LUNG_CANCER'].value_counts().plot(kind='pie', ax=axes[1],
    labels=['NO', 'YES'], autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Lung Cancer Proportion', fontsize=14)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Age Distribution by Cancer Status
fig, ax = plt.subplots(figsize=(10, 5))
df[df['LUNG_CANCER'] == 0]['AGE'].hist(alpha=0.6, bins=20, label='No Cancer', color='green', ax=ax)
df[df['LUNG_CANCER'] == 1]['AGE'].hist(alpha=0.6, bins=20, label='Cancer', color='red', ax=ax)
ax.set_title('Age Distribution by Cancer Status', fontsize=14)
ax.set_xlabel('Age')
ax.set_ylabel('Frequency')
ax.legend()
plt.show()

In [ ]:
# Feature Correlation Heatmap
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with Target Variable
target_corr = df.corr()['LUNG_CANCER'].drop('LUNG_CANCER').sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in target_corr.values]
target_corr.plot(kind='barh', color=colors)
plt.title('Feature Correlation with Lung Cancer', fontsize=14)
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Smoking vs Lung Cancer
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Smoking vs Cancer
pd.crosstab(df['SMOKING'], df['LUNG_CANCER']).plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Smoking vs Lung Cancer')
axes[0].set_xticklabels(['Non-Smoker', 'Smoker'], rotation=0)
axes[0].legend(['No Cancer', 'Cancer'])

# Gender vs Cancer
pd.crosstab(df['GENDER'], df['LUNG_CANCER']).plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Gender vs Lung Cancer')
axes[1].set_xticklabels(['Female', 'Male'], rotation=0)
axes[1].legend(['No Cancer', 'Cancer'])

# Chronic Disease vs Cancer
pd.crosstab(df['CHRONIC_DISEASE'], df['LUNG_CANCER']).plot(kind='bar', ax=axes[2], color=colors)
axes[2].set_title('Chronic Disease vs Lung Cancer')
axes[2].set_xticklabels(['No', 'Yes'], rotation=0)
axes[2].legend(['No Cancer', 'Cancer'])

plt.tight_layout()
plt.show()

## Step 5: Build Machine Learning Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

In [ ]:
# Train and evaluate all ML models
results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {'accuracy': acc, 'auc': auc, 'y_pred': y_pred, 'y_prob': y_prob}
    
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"AUC-ROC: {auc:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Cancer', 'Cancer']))

In [ ]:
# Confusion Matrix for each ML model
fig, axes = plt.subplots(1, 5, figsize=(24, 4))

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    axes[i].set_title(f'{name}\nAcc: {res["accuracy"]*100:.1f}%', fontsize=11)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices - ML Models', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves for ML Models
plt.figure(figsize=(10, 7))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - ML Models', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: Build ANN (Deep Learning) Model

In [ ]:
# Build ANN Model
ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification
])

ann_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

In [ ]:
# Train ANN
history = ann_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

print("\nANN Training Complete!")

In [ ]:
# ANN Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train Accuracy', marker='o', markersize=3)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s', markersize=3)
ax1.set_title('ANN - Accuracy over Epochs', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train Loss', marker='o', markersize=3)
ax2.plot(history.history['val_loss'], label='Validation Loss', marker='s', markersize=3)
ax2.set_title('ANN - Loss over Epochs', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate ANN
ann_loss, ann_acc = ann_model.evaluate(X_test, y_test, verbose=0)
ann_pred_prob = ann_model.predict(X_test, verbose=0).flatten()
ann_pred = (ann_pred_prob >= 0.5).astype(int)
ann_auc = roc_auc_score(y_test, ann_pred_prob)

print(f"ANN Test Accuracy: {ann_acc:.4f} ({ann_acc*100:.2f}%)")
print(f"ANN Test Loss: {ann_loss:.4f}")
print(f"ANN AUC-ROC: {ann_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, ann_pred, target_names=['No Cancer', 'Cancer']))

In [ ]:
# ANN Confusion Matrix
cm = confusion_matrix(y_test, ann_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Cancer', 'Cancer'],
            yticklabels=['No Cancer', 'Cancer'])
plt.title(f'ANN Confusion Matrix (Acc: {ann_acc*100:.1f}%)', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 7: Compare All Models

In [ ]:
# Add ANN results
results['ANN (Deep Learning)'] = {'accuracy': ann_acc, 'auc': ann_auc, 
                                   'y_pred': ann_pred, 'y_prob': ann_pred_prob}

# Create comparison DataFrame
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy (%)': [r['accuracy'] * 100 for r in results.values()],
    'AUC-ROC': [r['auc'] for r in results.values()]
}).sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)

print("\n" + "="*60)
print("  MODEL COMPARISON")
print("="*60)
print(comparison.to_string(index=False))
print("="*60)

best_model_name = comparison.iloc[0]['Model']
print(f"\n🏆 Best Model: {best_model_name} ({comparison.iloc[0]['Accuracy (%)']:.2f}%)")

In [ ]:
# Model Comparison Bar Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c']
bars = ax1.bar(comparison['Model'], comparison['Accuracy (%)'], color=colors_bar[:len(comparison)])
ax1.set_title('Model Accuracy Comparison', fontsize=14)
ax1.set_ylabel('Accuracy (%)')
ax1.set_ylim(0, 105)
for bar, acc in zip(bars, comparison['Accuracy (%)']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc:.1f}%', ha='center', fontsize=10)
ax1.tick_params(axis='x', rotation=30)

# AUC comparison
bars2 = ax2.bar(comparison['Model'], comparison['AUC-ROC'], color=colors_bar[:len(comparison)])
ax2.set_title('Model AUC-ROC Comparison', fontsize=14)
ax2.set_ylabel('AUC-ROC Score')
ax2.set_ylim(0, 1.1)
for bar, auc_val in zip(bars2, comparison['AUC-ROC']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{auc_val:.3f}', ha='center', fontsize=10)
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves - All Models Including ANN
plt.figure(figsize=(10, 7))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)', linewidth=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - All Models', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.title('Feature Importance (Random Forest)', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Step 8: Save Best Model

In [ ]:
# Save the Random Forest model (best traditional ML) and ANN model
with open('random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print("Random Forest model saved as random_forest_model.pkl")

ann_model.save('ann_model.h5')
print("ANN model saved as ann_model.h5")

# Save scaler and encoders
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved as scaler.pkl")

with open('label_encoders.pkl', 'wb') as f:
    pickle.dump({'gender': le_gender, 'cancer': le_cancer}, f)
print("Label encoders saved as label_encoders.pkl")

# Save training history
with open('training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)
print("Training history saved")

# Save model comparison
comparison.to_csv('model_comparison.csv', index=False)
print("Model comparison saved as model_comparison.csv")

print("\n" + "="*50)
print("PROJECT COMPLETE!")
print(f"Best ML Model: Random Forest")
print(f"Deep Learning: ANN ({ann_acc*100:.2f}%)")
print("="*50)

---

## Conclusion

- Built **5 Machine Learning models** and **1 Deep Learning (ANN) model** for lung cancer prediction
- Compared all models on **accuracy** and **AUC-ROC** metrics
- Key risk factors identified: **Smoking**, **Age**, **Shortness of Breath**, **Coughing**, **Chronic Disease**
- The models demonstrate that lung cancer risk can be predicted from patient survey data
- A **Streamlit web application** was built for real-time risk prediction

## Technologies Used
- **Python**, **TensorFlow/Keras**, **Scikit-Learn**
- **Pandas**, **NumPy**, **Matplotlib**, **Seaborn**
- **ML Models**: Logistic Regression, Decision Tree, Random Forest, SVM, KNN
- **DL Model**: Artificial Neural Network (ANN)
- **Streamlit** for web application